# **Revisiting Paradigms: One Problem, Four Solutions**

**DS112 — Computer Programming**

**Week 16:** · **Suggested class time: 5 hours (2-hour lecture + 3-hour laboratory)**

**ILO :** Compare imperative, functional, object-oriented, and declarative solutions to the same data science problem using Python, NumPy, and pandas.

In Week 3, you compared programming paradigms using pseudocode and analogies. Now that you know Python, NumPy, and pandas, we return to the same idea with a **real data science task** — and solve it **four times**, once per paradigm.

```{mermaid}
flowchart LR
    P["Simple linear regression"] --> I["Imperative<br/>step-by-step loops"]
    P --> F["Functional<br/>small pure functions"]
    P --> O["Object-oriented<br/>a model object"]
    P --> D["Declarative<br/>describe the model, let a library fit it"]
```

## **Task**

### Simple Linear Regression

Simple linear regression is used to model the relationship between two continuous variables. Often, the objective is to predict the value of an output variable (or response) based on the value of an input (or predictor) variable.

**Hypothesis: Assumes that the relationship between the independent and dependent variables can be described by a straight line.**

**Linear Regression function**

$$\hat{y} = mx + c\tag{1}$$

- $m$ is the slope of the line
- $c$ is the $y$ intercept
- $x$ is the independent variable

**Slope of Regression Line**

$$m = r\frac{\sigma_y}{\sigma_x}\tag{2}$$

- $r$ is the Pearson correlation coefficient
- $\sigma_y$ is the standard deviation of $y$
- $\sigma_x$ is the standard deviation of $x$

**$y$ intercept of a Regression Line**

$$c = \bar{y} - m\bar{x}\tag{3}$$

- $m$ is the slope of the regression line
- $\bar{y}$ is the mean of $y$ values
- $\bar{x}$ is the mean of $x$ values

**Pearson Correlation**
$$r = \frac{\sum((x-\bar{x})(y-\bar{y}))}{\sqrt{\sum(x-\bar{x})^2 \sum{(y-\bar{y})^2}}}\tag{4}$$

**Coefficient of determination** — how much of the variation in $y$ the line explains:

$$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}\tag{5}$$

## **The Data**

We use a **synthetic** advertising dataset shaped like the well-known *Advertising* data from *An Introduction to Statistical Learning*: TV advertising budget (in thousands) and product sales (in thousands of units) for 200 markets.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/advertising_synthetic.csv")
data = df[["TV", "Sales"]]
df.head()

## **Imperative Programming**

**Imperative programming** is a programming paradigm in which a program is expressed as a sequence of explicit instructions that tell the computer **how to perform a task step by step**. The programmer controls the flow of execution and typically changes the program's state through variables, assignments, loops, and conditional statements. Python naturally supports imperative programming through constructs such as `if`, `for`, `while`, variable assignment, and function calls. The programmer explicitly describes each operation the computer must perform. Thus, the central question in imperative programming is: **“What steps should I follow to solve the problem?”**

$$\bar{x} = \frac{\sum x_i}{n}$$

In [ ]:
xsum, ysum = 0, 0
count = 0
for _, (x, y) in data.iterrows():
    xsum += x
    ysum += y
    count += 1

xbar = xsum / count
ybar = ysum / count

$$\sigma = \sqrt{\frac{\sum (x_i - \bar{x})^2}{n - 1}}$$

> **Watch the parentheses.** An earlier version of this notebook wrote `(xstd / xcnt - 1) ** 0.5`, which divides by *n* and **then** subtracts 1. The sample standard deviation divides by **(n − 1)**: `(xstd / (count - 1)) ** 0.5`. Operator precedence matters!

In [ ]:
xss, yss = 0, 0
for _, (x, y) in data.iterrows():
    xss += (x - xbar) ** 2
    yss += (y - ybar) ** 2

xstd = (xss / (count - 1)) ** 0.5
ystd = (yss / (count - 1)) ** 0.5

$$r = \frac{\sum((x-\bar{x})(y-\bar{y}))}{\sqrt{\sum(x-\bar{x})^2 \sum{(y-\bar{y})^2}}}$$

In [ ]:
num = 0
den_x = 0
den_y = 0
for _, (x, y) in data.iterrows():
    num += (x - xbar) * (y - ybar)
    den_x += (x - xbar) ** 2
    den_y += (y - ybar) ** 2

r = num / ((den_x * den_y) ** 0.5)
m = r * ystd / xstd
c = ybar - m * xbar

ss_res, ss_tot = 0, 0
for _, (x, y) in data.iterrows():
    yhat = m * x + c
    ss_res += (y - yhat) ** 2
    ss_tot += (y - ybar) ** 2
r2 = 1 - ss_res / ss_tot

print(f"Imperative:  m = {m:.5f}, c = {c:.4f}, R² = {r2:.4f}")

Every quantity is built by a loop that **updates variables step by step**. It is easy to follow line by line, but long, repetitive, and slow (`iterrows` processes one row at a time).

## **Functional Programming**

**Functional programming** approaches computation primarily as the **evaluation and composition of functions**, emphasizing the transformation of input data into output data rather than repeatedly modifying shared program state. Functional programming commonly favors **pure functions**, immutable data, function composition, and functions that can be passed to or returned from other functions. Python is not a purely functional language, but it supports many functional programming concepts through first-class functions, `lambda` expressions, comprehensions, generators, and functions such as `map()`, `filter()`, and `reduce()`. In data processing, for instance, separate functions might clean, normalize, and transform a dataset, with the output of one function becoming the input of another rather than modifying global data. This approach is especially relevant to Python's data-processing ecosystem, where information frequently passes through a sequence of transformations. Its guiding question is: **“What transformations should I apply to the data?”**

In [ ]:
def mean(data):
    return sum(data) / len(data)


def std(data, ddof=1):
    xbar = mean(data)

    squared_diff = [
        (x - xbar) ** 2
        for x in data
    ]

    return (sum(squared_diff) / (len(data) - ddof)) ** 0.5


def corr(x, y):
    xbar = mean(x)
    ybar = mean(y)

    numerator = sum(
        (xi - xbar) * (yi - ybar)
        for xi, yi in zip(x, y)
    )

    denominator_x = sum(
        (xi - xbar) ** 2
        for xi in x
    )

    denominator_y = sum(
        (yi - ybar) ** 2
        for yi in y
    )

    return numerator / (denominator_x * denominator_y) ** 0.5


def r2(y, yhat):
    ybar = mean(y)

    ss_res = sum(
        (yi - yhati) ** 2
        for yi, yhati in zip(y, yhat)
    )

    ss_tot = sum(
        (yi - ybar) ** 2
        for yi in y
    )

    return 1 - (ss_res / ss_tot)


def fit(X, y, ddof=1):
    correlation = corr(X, y)
    xstd = std(X, ddof)
    ystd = std(y, ddof)

    coef = correlation * (ystd / xstd)
    intercept = mean(y) - coef * mean(X)

    return coef, intercept


def predict(X, model):
    coef, intercept = model

    return [
        intercept + coef * x
        for x in X
    ]


def score(X, y, model):
    yhat = predict(X, model)

    return r2(y, yhat)

In [ ]:
model = fit(data["TV"], data["Sales"])
print(f"Functional:  m = {model[0]:.5f}, c = {model[1]:.4f}, R² = {score(data['TV'], data['Sales'], model):.4f}")

Each function is **small and pure**: it receives values and returns a new value without modifying anything else. `fit` is built by **composing** `corr`, `std`, and `mean`, and the "model" is just a tuple of numbers passed from function to function.

## **Object-Oriented Programming**

**Object-oriented programming (OOP)** organizes a program around **objects that combine data with the behaviors that operate on that data**. Instead of viewing a program primarily as a sequence of instructions, OOP models the problem in terms of entities, their characteristics, responsibilities, and interactions with other entities. Python provides direct support for OOP through **classes, objects, attributes, methods, inheritance, encapsulation, and polymorphism**. For example, a data science application might define a `Dataset` class containing the dataset as its attributes and methods such as `clean()`, `summarize()`, and `visualize()`. Each object is then responsible for operations related to itself. This makes OOP particularly useful when a problem can be represented using identifiable entities with related data and behaviors. Its guiding question can therefore be expressed as: **“What objects are involved, and what can they do?”**

In [ ]:
class SimpleLinearRegression:
    '''A simple linear regression model with a scikit-learn-like interface.'''

    def __init__(self, ddof=1):
        self.ddof = ddof

    @staticmethod
    def _mean(values):
        return sum(values) / len(values)

    def _std(self, values):
        mu = self._mean(values)
        return (sum((v - mu) ** 2 for v in values) / (len(values) - self.ddof)) ** 0.5

    def fit(self, X, y):
        X, y = list(X), list(y)
        x_mean, y_mean = self._mean(X), self._mean(y)
        num = sum((xi - x_mean) * (yi - y_mean) for xi, yi in zip(X, y))
        den = (sum((xi - x_mean) ** 2 for xi in X) * sum((yi - y_mean) ** 2 for yi in y)) ** 0.5
        self.corr_ = num / den
        self.coef_ = self.corr_ * self._std(y) / self._std(X)
        self.intercept_ = y_mean - self.coef_ * x_mean
        return self

    def predict(self, X):
        return [self.intercept_ + self.coef_ * x for x in X]

    def score(self, X, y):
        y = list(y)
        y_mean = self._mean(y)
        ss_res = sum((yi - yh) ** 2 for yi, yh in zip(y, self.predict(X)))
        ss_tot = sum((yi - y_mean) ** 2 for yi in y)
        return 1 - ss_res / ss_tot


slr = SimpleLinearRegression().fit(data["TV"], data["Sales"])
print(f"OOP:         m = {slr.coef_:.5f}, c = {slr.intercept_:.4f}, R² = {slr.score(data['TV'], data['Sales']):.4f}")

The model is an **object** that keeps its learned values (`coef_`, `intercept_`) as attributes and offers **methods** (`fit`, `predict`, `score`). This version also fixes a performance issue in the original notebook, which recomputed the mean inside every loop iteration — an easy mistake that makes the code **quadratic** in the number of rows.

## **Declarative Programming**

**Declarative programming** focuses on describing **what result is desired rather than explicitly specifying every step required to produce it**. Instead of controlling the complete execution procedure, the programmer expresses rules, relationships, queries, or desired outcomes and allows another system or abstraction to determine how those results should be obtained. Python itself is primarily an imperative, object-oriented, and functional-capable language rather than a purely declarative language; however, **declarative programming can still be practiced through Python libraries and frameworks** that provide declarative interfaces. For example, a programmer can use a pandas expression such as `df.query("Sales > 1000")` to describe the records they want without manually specifying how every row should be traversed and checked. Similarly, Python ORMs allow database queries to be expressed through high-level model relationships rather than manually implementing the underlying database operations. Therefore, declarative Python shifts attention away from the procedure and toward the intended outcome, asking: **“What result do I want?”**

In [ ]:
from sklearn.linear_model import LinearRegression

X = df[["TV"]].to_numpy()
y = df["Sales"].to_numpy()

model = LinearRegression().fit(X, y)
print(f"Declarative: m = {model.coef_[0]:.5f}, c = {model.intercept_:.4f}, R² = {model.score(X, y):.4f}")

We **describe** the model we want (*a linear regression of Sales on TV*) and the library decides **how** to compute it.

### **A vectorized NumPy version**

NumPy lets us write the formulas almost exactly as they appear mathematically — another declarative style:

In [ ]:
x, y = df["TV"].to_numpy(), df["Sales"].to_numpy()
r = np.corrcoef(x, y)[0, 1]
m = r * y.std(ddof=1) / x.std(ddof=1)
c = y.mean() - m * x.mean()
r2 = 1 - ((y - (m * x + c)) ** 2).sum() / ((y - y.mean()) ** 2).sum()
print(f"NumPy:       m = {m:.5f}, c = {c:.4f}, R² = {r2:.4f}")

All five versions agree. They differ in **how the solution is organized**:

| | Imperative | Functional | Object-oriented | Declarative |
| - | ---------- | ---------- | --------------- | ----------- |
| **Organization** | loops updating variables | small composed functions | a model object with methods | a library call / formulas |
| **Readability** | step by step, but long | each piece is short and named | familiar `fit` / `predict` interface | shortest |
| **Testing** | hard to test parts separately | easy: each function is pure | test the object's methods | trust and verify the library |
| **Reuse** | copy and paste | import the functions | create many model objects | built in |
| **Speed** (200 rows → 1 million) | slow | slow (pure Python) | slow (pure Python) | **fast** (compiled code) |
| **Best for** | learning what the formulas do | pipelines and reusable math | systems with state and many models | everyday data science work |

## **Reflection**

* Which version helped you **understand** the formulas best? Which would you use in a **project**?
* Where did you see **state changing** (imperative) versus **new values being returned** (functional)?
* scikit-learn's `LinearRegression` is itself an **object**. How does knowing the OOP version change the way you read `model.fit(X, y)`?

## **Additional Notes: Paradigms in Other Languages**

### **Functional Style in JavaScript**

Created by **Brendan Eich** at **Netscape in 1995**. Originally designed to add **interactivity to web pages**

- **Major Milestones**
  - **1997** — Standardized as **ECMAScript**
  - **2000s** — AJAX enabled richer, more interactive web applications
  - **2009** — **Node.js** expanded JavaScript to server-side development
  - **2015** — **ES6** introduced `let`, `const`, arrow functions, classes, and modules

- **Applications**
  - Interactive web development
  - Backend/server development
  - Mobile & desktop applications
  - APIs & data visualization
  - Machine learning

- **Advantages**
  - Native support in web browsers
  - Large ecosystem
  - Cross-platform
  - **Multi-paradigm:** imperative, OOP, and functional

- **Disadvantages**
  - Dynamic typing can cause **runtime errors**
  - Large projects can become difficult to maintain
  - Dependency and tooling complexity
  - Asynchronous programming can add complexity


```javascript

// Transform
const doubled = numbers.map(x => x * 2);

// Select
const even = numbers.filter(x => x % 2 === 0);

// Aggregate
const total = numbers.reduce((sum, x) => sum + x, 0);

```

### **Declarative Programming in SQL**

Developed at **IBM in the 1970s** by Donald Chamberlin and Raymond Boyce. Based on **Edgar F. Codd’s relational model**. Originally called **SEQUEL**

- **Major Milestones**
  - **1986** — Became an ANSI standard
  - **1987** — Became an ISO standard
  - Continued evolving through later SQL standards

- **Applications**
  - Managing and querying **relational databases**
  - Retrieving, filtering, joining, and aggregating data
  - Used in systems such as MySQL, PostgreSQL, SQL Server, Oracle, and SQLite

- **Advantages**
  - **Declarative** — focuses on *what* result is needed
  - Powerful for structured and relational data
  - Widely standardized and supported

- **Disadvantages**
  - SQL dialects vary across database systems
  - Complex queries can be difficult to maintain
  - Not intended as a general-purpose programming language

```sql

SELECT name, salary
FROM employees
WHERE salary > 50000
ORDER BY salary DESC;

```

### **Imperative Programming in C**


```c
int numbers[] = {1, 2, 3, 4, 5};
int sum = 0;

for (int i = 0; i < 5; i++) {
    sum += numbers[i];
}

printf("%d", sum);
```

## **References**

Abelson, H., & Sussman, G. J. (1996). *Structure and interpretation of computer programs* (2nd ed.). MIT Press.

Buitinck, L., Louppe, G., Blondel, M., Pedregosa, F., Mueller, A., Grisel, O., Niculae, V., Prettenhofer, P., Gramfort, A., Grobler, J., Layton, R., VanderPlas, J., Joly, A., Holt, B., & Varoquaux, G. (2013). API design for machine learning software: Experiences from the scikit-learn project. *arXiv*. https://arxiv.org/abs/1309.0238

Codd, E. F. (1970). A relational model of data for large shared data banks. *Communications of the ACM, 13*(6), 377–387. https://doi.org/10.1145/362384.362685

Harris, C. R., Millman, K. J., van der Walt, S. J., Gommers, R., Virtanen, P., Cournapeau, D., Wieser, E., Taylor, J., Berg, S., Smith, N. J., Kern, R., Picus, M., Hoyer, S., van Kerkwijk, M. H., Brett, M., Haldane, A., del Río, J. F., Wiebe, M., Peterson, P., … Oliphant, T. E. (2020). Array programming with NumPy. *Nature, 585*(7825), 357–362. https://doi.org/10.1038/s41586-020-2649-2

Hughes, J. (1989). Why functional programming matters. *The Computer Journal, 32*(2), 98–107. https://doi.org/10.1093/comjnl/32.2.98

James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). *An introduction to statistical learning: With applications in Python*. Springer. https://doi.org/10.1007/978-3-031-38747-0

Maulion, M. R. (2022). *DS 112 computer programming* [Instructional material]. University of Science and Technology of Southern Philippines.

McKinney, W. (2022). *Python for data analysis: Data wrangling with pandas, NumPy, and Jupyter* (3rd ed.). O'Reilly Media. https://wesmckinney.com/book/

Sebesta, R. W. (2019). *Concepts of programming languages* (12th ed.). Pearson.